# 04 · Router label audit (§8.6, REQ-13)

Finding (2026-10-03, raw data):
- `call_transcripts.customer_text` holds **42 balance-inquiry templates**, spread evenly across the 6 categories.
- `detected_intents` = `consulta_general` (or null) in 100% of rows.
- `reason_category` = `contact_reason` (6 values) and escalation is ~10% in all of them: there is no signal.

→ The dataset gives no valid intent labels. The training corpus is team_generated
(`ml/data/intent_corpus.jsonl`, CON-02) and this audit is the evidence of why.

In [ ]:
from common import ROOT, connect, save_result, show

con = connect()

In [ ]:
templates = show(con, "Text diversity (digits masked)", """
    SELECT count(*) AS filas, count(DISTINCT regexp_replace(customer_text, '[0-9]+', '#', 'g')) AS plantillas
    FROM call_transcripts""")

In [ ]:
intents = show(con, "detected_intents", """
    SELECT detected_intents, count(*) AS n FROM call_transcripts GROUP BY 1 ORDER BY n DESC""")

In [ ]:
by_cat = show(con, "The same templates in every category", """
    SELECT i.reason_category, count(DISTINCT regexp_replace(t.customer_text, '[0-9]+', '#', 'g')) AS plantillas,
           count(*) AS n
    FROM call_transcripts t JOIN call_center_interactions i USING (interaction_id) GROUP BY 1 ORDER BY n DESC""")

## Team corpus and manual sample
With `ml/data/intent_corpus.jsonl`: distribution by class/language and agreement with `ml/labels/manual_labels.csv`.

In [ ]:
# analysis/ does not depend on sofia-ml: the files are read directly (same format as sofia_ml.labeling).
corpus_path = ROOT / "ml" / "data" / "intent_corpus.jsonl"
if corpus_path.exists():
    show(con, "Corpus by class and language", f"""
        SELECT label, language, origin, count(*) AS n, count(DISTINCT group_id) AS familias
        FROM read_json_auto('{corpus_path.as_posix()}') GROUP BY ALL ORDER BY label, language""")  # noqa: S608
else:
    print("no corpus yet:", corpus_path)

In [ ]:
save_result("label_audit", {
    "transcript_templates": templates.to_dicts(),
    "detected_intents": intents.to_dicts(),
    "by_category": by_cat.to_dicts(),
})